In [1]:
# Install required libraries
!pip install transformers datasets torch scikit-learn

In [2]:
# import required libraries
import pandas as pd
import numpy as np
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)

In [3]:
# sample dataset for text classification
data = {
    "text":[
        "I love this movie",
        "This film is terrible",
        "Amazing acting",
        "Worst experience",
        "Very good story",
        "Not worth watching",
        "Excellent direction",
        "Bad acting",
        "I enjoyed it",
        "Awful plot"
    ],
    "label":[1,0,1,0,1,0,1,0,1,0]
}
# dictionary to DataFrame
df = pd.DataFrame(data)
df.head()

,text,label
0,I love this movie,1
1,This film is terrible,0
2,Amazing acting,1
3,Worst experience,0
4,Very good story,1


In [4]:
# split dataset into training and validation sets
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['text'], df['label'], test_size=0.2, random_state=42
)
# load BERT tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [5]:
# tokenize training texts
train_encodings = tokenizer(
    train_texts.tolist(),
    truncation=True,
    padding=True
)
# tokenize validation texts
val_encodings = tokenizer(
    val_texts.tolist(),
    truncation=True,
    padding=True
)

In [6]:
# Custom Dataset class for PyTorch
class Dataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item
    def __len__(self):
        return len(self.labels)

In [7]:
# create dataset objects
train_dataset = Dataset(train_encodings, train_labels.tolist())
val_dataset = Dataset(val_encodings, val_labels.tolist())
# load pre-trained BERT model for sequence classification
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [8]:
# Define training arguments
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    num_train_epochs=2
)
# Initialize Trainer API
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)
# Fine-tune the BERT model
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=4, training_loss=0.6645623445510864, metrics={'train_runtime': 63.636, 'train_samples_per_second': 0.251, 'train_steps_per_second': 0.063, 'total_flos': 49333322880.0, 'train_loss': 0.6645623445510864, 'epoch': 2.0})

In [9]:
# Generate predictions
predictions = trainer.predict(val_dataset)
preds = np.argmax(predictions.predictions, axis=1)
# evaluate model performance
print("Accuracy:", accuracy_score(val_labels, preds))
print("Precision:", precision_score(val_labels, preds, zero_division=0))
print("Recall:", recall_score(val_labels, preds, zero_division=0))
print("F1 Score:", f1_score(val_labels, preds, zero_division=0))
# Display confusion matrix
print(confusion_matrix(val_labels, preds))

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Accuracy: 0.5
Precision: 0.5
Recall: 1.0
F1 Score: 0.6666666666666666
[[0 1]
 [0 1]]


## Observation

The dataset used for fine-tuning is small, which leads to limited model learning.
Because of this, evaluation metrics such as precision, recall, and F1 score are low.
With a larger Kaggle dataset, the model performance would improve significantly.
The fine-tuning pipeline, however, is correctly implemented.